# 🌍 Groq Multilingual Voice Chatbot

Speak in a language you choose, and the chatbot will **keep that language fixed for the entire turn**.

**Pipeline:** 🎤 Microphone → Whisper Large v3 (forced language) → Groq LLM (forced language) → gTTS (same language) → 🔊 Voice

Supported languages: English, Hindi, Kannada, Marathi, Tamil, Telugu.

## 1. Install dependencies

In [1]:
!pip -q install -U groq gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.


## 2. Add your Groq API keys to Colab Secrets

Open the 🔑 **Secrets** panel in Colab and add:

- `GROQ_API_KEY_STT` — used for Whisper
- `GROQ_API_KEY_LLM` — used for the chatbot LLM

Keeping them separate lets you use different keys for different parts of the system.

In [3]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY_STT = userdata.get("GROQ_API_KEY_STT")
GROQ_API_KEY_LLM = userdata.get("GROQ_API_KEY_LLM")

if not GROQ_API_KEY_STT:
    raise ValueError("Missing Colab Secret: GROQ_API_KEY_STT")
if not GROQ_API_KEY_LLM:
    raise ValueError("Missing Colab Secret: GROQ_API_KEY_LLM")

stt_client = Groq(api_key=GROQ_API_KEY_STT)
llm_client = Groq(api_key=GROQ_API_KEY_LLM)

STT_MODEL = "whisper-large-v3"
LLM_MODEL = "openai/gpt-oss-20b"

print("✅ Groq clients ready")

✅ Groq clients ready


## 3. Choose the language before speaking

Select one language. This choice is used for **Whisper, the LLM, and TTS**. Whisper will not try to detect the language.

In [30]:
LANGUAGES = {
    "English": "en",
    "Hindi": "hi",
    "Kannada": "kn",
    "Marathi": "mr",
    "Tamil": "ta",
    "Telugu": "te",
}

print("🌍 Available languages:\n")
for i, name in enumerate(LANGUAGES, 1):
    print(f"{i:2}. {name}")

choice = int(input("\nEnter the number of your language: "))

language_names = list(LANGUAGES.keys())
if choice < 1 or choice > len(language_names):
    raise ValueError("Invalid language number")

SELECTED_LANGUAGE_NAME = language_names[choice - 1]
SELECTED_LANGUAGE = LANGUAGES[SELECTED_LANGUAGE_NAME]

print(f"\n✅ Selected language: {SELECTED_LANGUAGE_NAME}")
print(f"🔤 Code: {SELECTED_LANGUAGE}")

🌍 Available languages:

 1. Bengali
 2. English
 3. Gujarati
 4. Hindi
 5. Kannada
 6. Malayalam
 7. Marathi
 8. Punjabi
 9. Tamil
10. Telugu

Enter the number of your language: 10

✅ Selected language: Telugu
🔤 Code: te


## 4. Browser microphone recorder

Run this once. It records directly from your browser microphone, so there is no file upload.

In [5]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def record_audio(seconds=8, filename="recording.webm"):
    display(Javascript(r'''
    async function recordAudio(seconds) {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, seconds * 1000));
      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      const blob = new Blob(chunks, {type: 'audio/webm'});
      const reader = new FileReader();
      return await new Promise(resolve => {
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
      });
    }
    '''))
    data = eval_js(f"recordAudio({seconds})")
    audio = b64decode(data.split(",")[1])
    with open(filename, "wb") as f:
        f.write(audio)
    return filename

## 5. 🎤 Talk to the chatbot

Run this cell for each turn. The selected language is fixed — there is **no automatic language detection**.

In [31]:
import time
from gtts import gTTS
from IPython.display import Audio, display

RECORD_SECONDS = 4

print(f"🎤 Speak now in {SELECTED_LANGUAGE_NAME} ({RECORD_SECONDS} seconds)...")
audio_path = record_audio(RECORD_SECONDS)

# ---------- STT ----------
with open(audio_path, "rb") as f:
    start = time.perf_counter()
    transcription = stt_client.audio.transcriptions.create(
        file=(audio_path, f.read()),
        model=STT_MODEL,
        language=SELECTED_LANGUAGE,
        response_format="json",
        temperature=0.0,
    )
    stt_time = time.perf_counter() - start

transcript = transcription.text.strip()

print(f"\n📝 You: {transcript}")
print(f"🌍 Language: {SELECTED_LANGUAGE_NAME}")
print(f"⏱️ STT latency: {stt_time:.2f}s")

# ---------- LLM ----------
system_prompt = f"""
You are a friendly multilingual voice chatbot.

The user selected {SELECTED_LANGUAGE_NAME} as the conversation language.

STRICT RULES:
1. Respond ONLY in {SELECTED_LANGUAGE_NAME}.
2. Do not switch to English or another language.
3. Do not translate unless the user explicitly asks for a translation.
4. Use natural conversational language.
5. Keep responses concise because they will be spoken aloud.
"""

print("\n🤖 Thinking...")
start = time.perf_counter()

response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": transcript},
    ],
    reasoning_effort="medium",
    temperature=0.7,
    max_completion_tokens=500,
)

llm_time = time.perf_counter() - start
answer = response.choices[0].message.content.strip()

print(f"\n🤖 Bot: {answer}")
print(f"⏱️ LLM latency: {llm_time:.2f}s")

# ---------- TTS ----------
print(f"\n🔊 Speaking in {SELECTED_LANGUAGE_NAME}...")
gTTS(text=answer, lang=SELECTED_LANGUAGE).save("bot_response.mp3")
display(Audio("bot_response.mp3", autoplay=True))

🎤 Speak now in Telugu (4 seconds)...


<IPython.core.display.Javascript object>


📝 You: మీరు ఎల్లా ఉన్నారు?
🌍 Language: Telugu
⏱️ STT latency: 0.42s

🤖 Thinking...

🤖 Bot: నేను బాగున్నాను, ధన్యవాదాలు! మీరు ఎలా ఉన్నారు?
⏱️ LLM latency: 0.29s

🔊 Speaking in Telugu...


## 6. Change language

If you want to switch languages, simply rerun the **language selection cell (Cell 3)** and choose another number. Then run the conversation cell again.

Example:
- Select **Hindi** → Whisper `hi` → LLM Hindi → TTS `hi`
- Select **Marathi** → Whisper `mr` → LLM Marathi → TTS `mr`
- Select **Urdu** → Whisper `ur` → LLM Urdu → TTS `ur`

This avoids Whisper's automatic language-detection errors on short recordings.

## 7. Next upgrade

Once this fixed-language version is reliable, the next step is to make it feel like a real chatbot: a **Record / Stop button**, persistent conversation memory, and a cleaner chat interface.